## <font color="#FF9900">Pipeline Bronze AWS</font>

> "Notebook criado para recuperar os arquivos em CSV na pasta raw, converter em bytes no formato Parquet e enviar para o bucket da AWS S3 na camada bronze."

In [ ]:
%load_ext autoreload
%autoreload 2


from IPython import get_ipython
import pandas as pd 
import os
import dotenv
from pathlib import Path
from pandas import DataFrame
from botocore.exceptions import ClientError

# Instala o boto3 biblioteca utilizada para conexão do aws 
try:
    import boto3
except ModuleNotFoundError:
    get_ipython().run_line_magic('pip', 'install boto3')
    import boto3

dotenv.load_dotenv()

# Trata o pacote local 'src' para conseguir extrair as funções uteis
try:
    from src.data.utils import gerar_df_dic, converter_para_parquet_bytes, salvar_parquet_local,salvar_parquet_s3
except ModuleNotFoundError:
    get_ipython().run_line_magic('pip', 'install -e ..')
    from src.data.utils import gerar_df_dic, converter_para_parquet_bytes, salvar_parquet_local,salvar_parquet_s3

In [ ]:
# Recuperar as variaveis para acessar o AWS 
AWS_REGION = os.getenv("AWS_REGION")
aws_access_key_id=os.getenv("AWS_ACESS_KEY_ID")
aws_secret_access_key=os.getenv("AWS_SECRET_ACESS_KEY")
#aws_session_token=os.getenv("aws_session_token") Token não é nessário fora do AWS Academy

# Gerar conexão com o S3 
boto3.setup_default_session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
   # aws_session_token=aws_session_token,  # <--- Não esqueça de pegar este lá no "AWS Details"
    region_name=AWS_REGION
)

configurado o boto 3 iremos utiliza-lo para acessar o nosso AWS atraves das nossas variaveis, iremos verificar se já existe o bucket da fiap-pos-tech-002, caso nao exista iremos cria-lo, se existir só iremos conectar a ele

In [ ]:
s3 = boto3.client('s3')
nome_do_bucket = os.getenv("BUCKET_NAME")

response = s3.list_buckets()
buckets_existentes = [bucket['Name'] for bucket in response['Buckets']]

try:
    # verifica a existencia do bucket da Fiap na AWS conectada
    s3.head_bucket(Bucket=nome_do_bucket)
    print(f"Bucket '{nome_do_bucket}' já existe.")
except ClientError as e:
    # Se der erro 404 (Not Found), significa que o bucket não existe então iremos cria-lo, 
    # se retornar o erro 403 precisaremos mudar o nome do BUCKET em .env
    error_code = e.response['Error']['Code']
    if error_code == '404':
        s3.create_bucket(Bucket=nome_do_bucket)
        print(f"Bucket '{nome_do_bucket}' criado com sucesso.")
    else:
        # Outro erro
        print(f"Erro ao acessar o bucket: {e}")

Utilizamos a função gerar_df_dic localizada no src\data\utils.py 
a função é utilizada para carregar os csv no Raw e carregar todos os dataframes para que possamos transforma-los em .parquet

## Camada BRONZE

Importante a Base RAW dicionários vem com acento "DICIONÁRIO" é necessário remover o acento para conseguir processar. fizemos isso para rodar no MAC e Windows 

## <font color="#FF9900">Pipeline de Ingestão AWS S3</font>

In [ ]:
#Para Rodar na AWS
s3 = boto3.client('s3')
nome_do_bucket = os.getenv("BUCKET_NAME")

anos = [2023, 2024, 2025] 
tabelas = ['TS_ALUNO', 'TS_ITEM', 'TS_ESTADO', 'TS_MUNICIPIO']

for ano in anos: 
    for tabela in tabelas:
        print(f'\n--- Trabalhando com a tabela {tabela} ({ano}) ---')
        
        # 3. Carrega os dataframes originais
        df, dic = gerar_df_dic(ano, tabela)
        
        # 4. Converte os DataFrames para bytes em formato Parquet
        df_bytes = converter_para_parquet_bytes(df)
        dic_bytes = converter_para_parquet_bytes(dic)
        
        # 5. Define as chaves (caminhos de pastas) estruturadas no S3
        chave_df = f"bronze/ano={ano}/dados/{tabela}.parquet"
        chave_dicionario = f"bronze/ano={ano}/dicionario/dicionario_{tabela}.parquet"
        
        # 6. Salva no S3
        print(f'Enviando dados de {tabela} para o S3...')
        salvar_parquet_s3(s3, nome_do_bucket, chave_df, df_bytes)
        
        print(f'Enviando dicionário de {tabela} para o S3...')
        salvar_parquet_s3(s3, nome_do_bucket, chave_dicionario, dic_bytes)

## <font color="#FF9900">AWS S3</font>
<font size="3" color="gray">Evidências de execução da criação da camada Bronze</font>



![Status AWS](https://img.shields.io/badge/AWS-S3_Bronze_Created-green?style=for-the-badge&logo=amazonaws)


![Bucket Criado no S3](../reports/figures/bucket_criado.png)





![Bucket Criado no S3](../reports/figures/pasta_bronze.png)


![Bucket Criado no S3](../reports/figures/particoes_ano.png)


<font color="green" size="5"> Particoes por ano </font>

 
![Bucket Criado no S3](../reports/figures/header_s3_bronze.png)

<font color="green" size="5"> Dados </font>

![Bucket Criado no S3](../reports/figures/pasta_dados.png)

<font color="green" size="5"> Dicionarios </font>

![Bucket Criado no S3](../reports/figures/dicionario.png)

